# Rabin-Karp Algorithm Implementation for Spam Detection

In [ ]:
import pandas as pd
import re
import random
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

### Load the CSV

In [30]:
df = pd.read_csv("spam.csv", encoding="latin1")

df = df[["v1", "v2"]]
df.columns = ["label", "text"]

df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Train & Test Split

In [31]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

len(train_df), len(test_df)

(4457, 1115)

### Normalization + Tokenization Functions

In [ ]:
def normalize(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def get_tokens(text: str):
    return normalize(text).split()

### Learn Spam Keywords From Training Data

In [51]:
STOPWORDS = {
    "a", "an", "the", "to", "and", "or", "of", "in", "on", "for", "is",
    "it", "this", "that", "you", "me", "i", "we", "are", "be", "your",
    "with", "at", "from", "by", "my", "our", "us"
}

spam_tokens = Counter()
ham_tokens = Counter()

for _, row in train_df.iterrows():
    label = row["label"]
    tokens = set(get_tokens(row["text"]))
    if label == "spam":
        spam_tokens.update(tokens)
    else:
        ham_tokens.update(tokens)

min_spam_count = 18
min_spam_ratio = 0.80

spam_keywords = []

for token, sp_count in spam_tokens.items():
    # 1) basic quality filters
    if len(token) < 4:        # too short → often noisy
        continue
    if token.isdigit():       # pure numbers are often not helpful
        continue
    if token in STOPWORDS:    # basic stopword removal
        continue

    ham_count = ham_tokens[token]
    total = sp_count + ham_count
    if total == 0:
        continue

    ratio = sp_count / total
    if sp_count >= min_spam_count and ratio >= min_spam_ratio:
        spam_keywords.append(token)

spam_keywords = sorted(spam_keywords)
print(spam_keywords)


['150p', '150ppm', 'apply', 'await', 'awarded', 'camera', 'cash', 'claim', 'code', 'collection', 'contact', 'customer', 'draw', 'guaranteed', 'http', 'landline', 'latest', 'line', 'mobile', 'national', 'nokia', 'orange', 'prize', 'rate', 'receive', 'ringtone', 'selected', 'service', 'shows', 'tone', 'urgent', 'video', 'vouchers', 'weekly']


### Rabin–Karp Substring Search Implementation

In [52]:
def rabin_karp_search(text: str, pattern: str, base: int = 257, mod: int = 10**9 + 7) -> bool:
    """
    Return True if 'pattern' is found as substring in 'text'.
    """
    n, m = len(text), len(pattern)
    if m == 0 or m > n:
        return False

    power = 1
    for _ in range(m - 1):
        power = (power * base) % mod

    h_text = 0
    h_pat = 0
    for i in range(m):
        h_text = (h_text * base + ord(text[i])) % mod
        h_pat = (h_pat * base + ord(pattern[i])) % mod

    if h_text == h_pat and text[:m] == pattern:
        return True

    for i in range(m, n):
        h_text = (h_text - ord(text[i - m]) * power) % mod
        h_text = (h_text * base + ord(text[i])) % mod
        h_text %= mod

        if h_text == h_pat:
            if text[i - m + 1 : i + 1] == pattern:
                return True

    return False

### Spam Detection Using Rabin–Karp

In [53]:
def contains_spam_keyword(message: str, patterns=spam_keywords) -> bool:
    text_norm = normalize(message)
    for pat in patterns:
        if rabin_karp_search(text_norm, pat):
            return True
    return False

def predict_label(message: str) -> str:
    return "spam" if contains_spam_keyword(message) else "ham"

### Evaluate on the Test Set

In [54]:
y_true = list(test_df["label"])
y_pred = [predict_label(text) for text in test_df["text"]]

print(classification_report(y_true, y_pred))


              precision    recall  f1-score   support

         ham       0.97      0.96      0.97       966
        spam       0.77      0.81      0.79       149

    accuracy                           0.94      1115
   macro avg       0.87      0.88      0.88      1115
weighted avg       0.94      0.94      0.94      1115



### Example Predictions

In [38]:
examples = [
    "Win a FREE prize now!!! Text WIN to 54321",
    "Are we still meeting for lunch today?",
    "URGENT! You have won a 1 week FREE vacation. Call now!",
    "Can you send me the homework?",
]

for msg in examples:
    print("Message:", msg)
    print("Predicted:", predict_label(msg))
    print()


Message: Win a FREE prize now!!! Text WIN to 54321
Predicted: spam

Message: Are we still meeting for lunch today?
Predicted: ham

Message: URGENT! You have won a 1 week FREE vacation. Call now!
Predicted: spam

Message: Can you send me the homework?
Predicted: ham

